In [3]:
# Load all relevant files and count the number of matching question IDs between prediction files and the gold file.
import json

# Load the gold truth
with open("../data/BioASQ-task13bPhaseA-testset4.txt") as f:
    gold_data = json.load(f)
gold_questions = gold_data["questions"]
gold_dict = {q["id"]: q.get("documents", []) for q in gold_questions}

# Load predictions
filepaths = {
   "neural": "BioASQ-task13b-phaseA-testset4-neural-results.json",
    "traditional_w2v": "../output/results_traditional_model_w2v.json",
    "traditional_basic": "../output/4-The Relevants-traditional-model.json"
}

# Initialize results
match_stats = {}

for name, path in filepaths.items():
    with open(path) as f:
        pred_data = json.load(f)
    pred_dict = {q["id"]: q.get("documents", []) for q in pred_data["questions"]}

    # Find matching question IDs
    common_ids = set(gold_dict.keys()) & set(pred_dict.keys())

    # Count how many predictions are non-empty and match gold questions
    non_empty_preds = sum(1 for qid in common_ids if pred_dict[qid])
    non_empty_gold = sum(1 for qid in common_ids if gold_dict[qid])
    non_empty_both = sum(1 for qid in common_ids if pred_dict[qid] and gold_dict[qid])

    match_stats[name] = {
        "total_in_gold": len(gold_dict),
        "total_in_pred": len(pred_dict),
        "matching_ids": len(common_ids),
        "pred_with_docs": non_empty_preds,
        "gold_with_docs": non_empty_gold,
        "both_have_docs": non_empty_both
    }

match_stats


{'neural': {'total_in_gold': 85,
  'total_in_pred': 85,
  'matching_ids': 85,
  'pred_with_docs': 63,
  'gold_with_docs': 0,
  'both_have_docs': 0},
 'traditional_w2v': {'total_in_gold': 85,
  'total_in_pred': 85,
  'matching_ids': 85,
  'pred_with_docs': 63,
  'gold_with_docs': 0,
  'both_have_docs': 0},
 'traditional_basic': {'total_in_gold': 85,
  'total_in_pred': 85,
  'matching_ids': 85,
  'pred_with_docs': 63,
  'gold_with_docs': 0,
  'both_have_docs': 0}}

In [4]:
import json
from sklearn.metrics import precision_score, recall_score, f1_score

# Load files
with open('BioASQ-task13b-phaseA-testset4-neural-results.json') as f:
    predictions = json.load(f)

with open('../data/BioASQ-training13b/training13b.json') as f:
    gold = json.load(f)

# Convert gold answers to dictionary by ID
gold_dict = {q["id"]: q for q in gold["questions"]}

# Initialize evaluation counters
yesno_correct = 0
factoid_total = 0
factoid_correct = 0
list_precisions = []
list_recalls = []
list_f1s = []

for pred in predictions["questions"]:
    qid = pred["id"]
    if qid not in gold_dict:
        continue

    g = gold_dict[qid]
    qtype = g["type"]
    
    if qtype == "yesno":
        if pred["exact_answer"].strip().lower() == g["exact_answer"].strip().lower():
            yesno_correct += 1

    elif qtype == "factoid":
        factoid_total += 1
        pred_ans = set(a.lower() for a in pred["exact_answer"])
        gold_ans = set(a.lower() for group in g["exact_answer"] for a in group)
        if pred_ans & gold_ans:
            factoid_correct += 1

    elif qtype == "list":
        pred_ans = set(a.lower() for group in pred["exact_answer"] for a in group)
        gold_ans = set(a.lower() for group in g["exact_answer"] for a in group)
        tp = len(pred_ans & gold_ans)
        precision = tp / len(pred_ans) if pred_ans else 0
        recall = tp / len(gold_ans) if gold_ans else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
        list_precisions.append(precision)
        list_recalls.append(recall)
        list_f1s.append(f1)

# Report
print(f"Yes/No Accuracy: {yesno_correct} / {len([q for q in predictions['questions'] if gold_dict[q['id']]['type'] == 'yesno'])}")
print(f"Factoid Accuracy: {factoid_correct} / {factoid_total}")
print(f"List - Precision: {sum(list_precisions)/len(list_precisions):.2f}")
print(f"List - Recall: {sum(list_recalls)/len(list_recalls):.2f}")
print(f"List - F1: {sum(list_f1s)/len(list_f1s):.2f}")

KeyError: '67e6cf2618b1e36f2e0000d0'